In [2]:
import kagglehub
import pandas as pd
import os

path = kagglehub.dataset_download("adizafar/large-random-tweets-from-pakistan")
csv_files = [f for f in os.listdir(path) if f.endswith('.csv')]
file_path = os.path.join(path, csv_files[0])


df = pd.read_csv(file_path, encoding='latin-1')

text_col = 'full_text' if 'full_text' in df.columns else 'text' if 'text' in df.columns else df.columns[0]
display(df[[text_col]].head())

C:\Users\azozk\AppData\Local\Temp\ipykernel_26556\2772234466.py:10: DtypeWarning: Columns (5: reply_count) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path, encoding='latin-1')


,full_text
0,ØªÛØ±Ø§ ÙÛÚØ± Ù ÛØ±Ø§ ÙÛÚØ± ÙÙØ§Ø² Ø...
1,"Happy birthday to my brother n boss , May you ..."
2,â¤ï¸â¤ï¸
3,`suspicious Â°jikook au jimin'in yaÅadÄ±ÄÄ± ...
4,Speaking of @SpiderMan... ð https://t.co/...


In [3]:
import re
import emoji

def clean_tweet(text):
    if not isinstance(text, str):
        return ""
  
    text = re.sub(r'\bRT\b', '', text)
  
    text = re.sub(r'@\w+', '', text)
    
    text = re.sub(r'#\w+', '', text)
    
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    
    text = emoji.replace_emoji(text, replace='')
    
    text = re.sub(r'\s+', ' ', text).strip().lower()
    return text


df['clean_text'] = df['full_text'].apply(clean_tweet)

print("--- Cleaned Tweets ---")
display(df[['clean_text']].head(10))

--- Cleaned Tweets ---


,clean_text
0,øªûø±ø§ ùûúø± ù ûø±ø§ ùûúø± ùùø§ø² ø...
1,"happy birthday to my brother n boss , may you ..."
2,â¤ï¸â¤ï¸
3,`suspicious â°jikook au jimin'in yaåadä±ää± ...
4,speaking of ... ð
5,"alexa, skip to 2021"
6,jb tm moon lover ho gy to sooraj to tappy ga n...
7,alexa skip to 2022 now
8,"but this was one of my first shoots, about 6 y..."
9,trying phoneix in 140 ping ð¥


In [4]:
import nltk
from nltk.tokenize import word_tokenize
from nltk.lm.preprocessing import padded_everygram_pipeline
from nltk.lm import MLE

nltk.download('punkt')
nltk.download('punkt_tab')


corpus = list(df['clean_text'].dropna())[:10000] 
tokenized_text = [word_tokenize(sentence) for sentence in corpus if sentence]


n = 2
train_data, padded_sents = padded_everygram_pipeline(n, tokenized_text)

lm = MLE(n)
lm.fit(train_data, padded_sents)

print(f"Bi-gram MLE Model successfully trained! Vocabulary size: {len(lm.vocab)}")

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\azozk\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt.zip.
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\azozk\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


Bi-gram MLE Model successfully trained! Vocabulary size: 24795


In [5]:

prob = lm.score('is', ['pakistan'])
print(f"Probability of 'is' given 'pakistan': {prob:.6f}")


perp = lm.perplexity([('pakistan',)])
print(f"Perplexity of the word 'pakistan': {perp:.2f}")

Probability of 'is' given 'pakistan': 0.036496
Perplexity of the word 'pakistan': 1018.26
